# 2 · Empfehlungen: beliebt oder zusammen gekauft?
**Päckli & Co.** · Data Science für Marketing · FHNW CAS Data Driven Marketing

Jede E-Mail der Kampagne enthält bis zu drei Produktempfehlungen. Gute Empfehlungen bringen Käufe, schlechte bringen
Abmeldungen. Ihr vergleicht zwei Ansätze und prüft, welcher besser ist, ohne einen einzigen Kunden anzuschreiben.

**Minimum:** drei Empfehlungen pro Kunde (`empfehlungen`) und die Trefferquote beider Ansätze.
**Erweiterung (optional):** bereits gekaufte Produkte ausschliessen, Beliebtheit herausrechnen, ausgelistete Produkte.

**So arbeitet ihr in jedem Schritt**
1. Lest das **Ziel**.
2. Kopiert den **Prompt** ins Gemini-Fenster (Gemini-Symbol oben rechts) und lasst euch den Code erklären.
3. Fügt den Code in die leere Zelle ein und führt sie aus (▶ links oder `Shift + Enter`).
4. **Plausibel?** Prüft das Ergebnis mit gesundem Menschenverstand.
5. Steckt ihr fest, führt die **💡 Lösung** aus. Das ist keine Schande: Die Lösung liefert dieselben Tabellen wie euer Code.

In [ ]:
#@title ▶ Setup: tragt eure Teamnummer ein und führt diese Zelle zuerst aus
TEAM = 0  #@param {type:"integer"}
BASIS_URL = "https://raw.githubusercontent.com/badrustanicki/fhnw-ds-marketing-2026/main/"
import os, urllib.request
if not os.path.exists("kurs.py"):
    urllib.request.urlretrieve(BASIS_URL + "kurs.py", "kurs.py")
import duckdb, numpy as np, pandas as pd, matplotlib.pyplot as plt
from kurs import *
start(TEAM, BASIS_URL)

In [ ]:
#@title ▶ Aufholen: Ergebnisse aus Block 1 laden (eure eigenen, sonst das Musterergebnis)
orders_clean = aufholen("orders_clean")
rfm = aufholen("rfm")

### 1 Beliebt

**Ziel:** Die drei Produkte finden, die die meisten verschiedenen Kunden gekauft haben. Das ist der einfache Ansatz: allen dasselbe empfehlen.

**Prompt für Gemini** (kopieren und im Gemini-Fenster einfügen):
```text
Welche 3 Produkte haben in orders_clean die meisten verschiedenen Kunden gekauft? Zeige stock_code, description aus products und die Anzahl Kunden. Speichere die drei stock_codes in einer Python-Liste top3.
```

In [ ]:
# Euer Code: fügt hier ein, was Gemini vorschlägt, und führt die Zelle aus (▶).

In [ ]:
#@title 💡 Lösung: nur wenn ihr nicht weiterkommt (Doppelklick zeigt den Code, ▶ führt ihn aus)
beliebt = duckdb.sql("""
SELECT o.stock_code, p.description, COUNT(DISTINCT o.customer_id) AS kunden
FROM orders_clean o JOIN products p ON p.stock_code = o.stock_code
GROUP BY o.stock_code, p.description
ORDER BY kunden DESC, o.stock_code
LIMIT 3
""").df()
top3 = beliebt["stock_code"].tolist()
beliebt

**Plausibel?** Sind es echte Produkte, kein Porto? Wenn doch: Ist orders_clean wirklich sauber?

### 2 Was wird zusammen gekauft?

**Ziel:** Für ein Produkt sehen, welche Produkte am häufigsten auf derselben Rechnung stehen.

**Prompt für Gemini** (kopieren und im Gemini-Fenster einfügen):
```text
Welche Produkte stehen in orders_clean am häufigsten auf derselben Rechnung wie P1006? Zeige die Top 8 mit description und der Anzahl gemeinsamer Rechnungen.
```

In [ ]:
# Euer Code: fügt hier ein, was Gemini vorschlägt, und führt die Zelle aus (▶).

In [ ]:
#@title 💡 Lösung: nur wenn ihr nicht weiterkommt (Doppelklick zeigt den Code, ▶ führt ihn aus)
duckdb.sql("""
WITH mit_x AS (SELECT DISTINCT invoice FROM orders_clean WHERE stock_code = 'P1006')
SELECT p.description AS auch_gekauft, COUNT(DISTINCT o.invoice) AS gemeinsam
FROM mit_x JOIN orders_clean o ON o.invoice = mit_x.invoice
JOIN products p ON p.stock_code = o.stock_code
WHERE o.stock_code <> 'P1006'
GROUP BY p.description ORDER BY gemeinsam DESC LIMIT 8
""")

**Plausibel?** Fällt euch ein Muster auf, z. B. dieselbe Serie in anderen Farben? Probiert ein anderes Produkt, etwa P1084.

### 3 Zusammen gekauft, für jeden Kunden

**Ziel:** Für jeden Kunden die drei Produkte finden, die am häufigsten zusammen mit dem gekauft werden, was er schon gekauft hat. «Wer A kaufte, kaufte auch B.»

**Prompt für Gemini** (kopieren und im Gemini-Fenster einfügen):
```text
Schreibe eine Python-Funktion zusammen_gekauft(tabelle), die mit DuckDB für jeden Kunden drei Empfehlungen berechnet: 1) Zähle für jedes Produktpaar, auf wie vielen Rechnungen beide stehen. 2) Gib jedem Produkt pro Kunde Punkte: die Summe dieser Zähler über alle Produkte, die der Kunde gekauft hat. 3) Nimm pro Kunde die 3 Produkte mit den meisten Punkten. Ergebnis: DataFrame mit customer_id, rec_1, rec_2, rec_3. Rufe die Funktion mit 'orders_clean' auf und speichere das Ergebnis als empfehlungen.
```

In [ ]:
# Euer Code: fügt hier ein, was Gemini vorschlägt, und führt die Zelle aus (▶).

In [ ]:
#@title 💡 Lösung: nur wenn ihr nicht weiterkommt (Doppelklick zeigt den Code, ▶ führt ihn aus)
def zusammen_gekauft(tabelle):
    return duckdb.sql(f"""
    WITH rechnungen AS (SELECT DISTINCT invoice, stock_code FROM {tabelle}),
    paare AS (       -- auf wie vielen Rechnungen stehen zwei Produkte gemeinsam?
        SELECT a.stock_code AS produkt, b.stock_code AS empfehlung, COUNT(*) AS gemeinsam
        FROM rechnungen a JOIN rechnungen b ON a.invoice = b.invoice AND a.stock_code <> b.stock_code
        GROUP BY a.stock_code, b.stock_code),
    gekauft AS (SELECT DISTINCT customer_id, stock_code FROM {tabelle}),
    punkte AS (      -- Punkte pro Kunde und Produkt
        SELECT g.customer_id, p.empfehlung, SUM(p.gemeinsam) AS punkte
        FROM gekauft g JOIN paare p ON p.produkt = g.stock_code
        GROUP BY g.customer_id, p.empfehlung),
    rang AS (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY punkte DESC, empfehlung) AS r FROM punkte)
    SELECT customer_id,
           MAX(CASE WHEN r = 1 THEN empfehlung END) AS rec_1,
           MAX(CASE WHEN r = 2 THEN empfehlung END) AS rec_2,
           MAX(CASE WHEN r = 3 THEN empfehlung END) AS rec_3
    FROM rang WHERE r <= 3
    GROUP BY customer_id
    """).df()

empfehlungen = rfm[["customer_id"]].merge(zusammen_gekauft("orders_clean"), on="customer_id", how="left")
for i, k in enumerate(["rec_1", "rec_2", "rec_3"]):   # wer keine Paare hat, bekommt die beliebtesten
    empfehlungen[k] = empfehlungen[k].fillna(top3[i])
empfehlungen.head()

**Plausibel?** Schaut euch zwei, drei Kunden an: Was haben sie gekauft, was wird empfohlen? Passt das zusammen? Bekommen alle dasselbe, oder ist es persönlich?

In [ ]:
plausibel("empfehlungen", empfehlungen)

### 4 Welcher Ansatz ist besser? Offline-Evaluation

**Ziel:** Die letzte Bestellung jedes Kunden verstecken, aus der Vorgeschichte empfehlen und zählen, wie oft eine Empfehlung in der versteckten Bestellung vorkommt: die Trefferquote. Für beide Ansätze.

**Prompt für Gemini** (kopieren und im Gemini-Fenster einfügen):
```text
Offline-Evaluation: 1) Finde für jeden Kunden mit mindestens 2 Rechnungen in orders_clean die letzte Rechnung (nach invoice_date). 2) Erstelle in DuckDB eine Tabelle training = orders_clean ohne diese letzten Rechnungen. 3) Empfehle aus training: einmal mit zusammen_gekauft('training'), einmal die 3 beliebtesten Produkte aus training für alle. 4) Trefferquote: Anteil dieser Kunden, bei denen mindestens eine Empfehlung in ihrer letzten Rechnung vorkommt. Vergleiche beide Ansätze.
```

In [ ]:
# Euer Code: fügt hier ein, was Gemini vorschlägt, und führt die Zelle aus (▶).

In [ ]:
#@title 💡 Lösung: nur wenn ihr nicht weiterkommt (Doppelklick zeigt den Code, ▶ führt ihn aus)
duckdb.sql("""
CREATE OR REPLACE TABLE letzte AS          -- letzte Rechnung pro Kunde (mit mindestens 2 Rechnungen)
SELECT customer_id, invoice FROM (
    SELECT customer_id, invoice,
           ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY MAX(invoice_date) DESC, invoice DESC) AS r,
           COUNT(*) OVER (PARTITION BY customer_id) AS rechnungen
    FROM orders_clean GROUP BY customer_id, invoice)
WHERE r = 1 AND rechnungen >= 2
""")
duckdb.sql("CREATE OR REPLACE TABLE training AS SELECT * FROM orders_clean WHERE invoice NOT IN (SELECT invoice FROM letzte)")
versteckt = duckdb.sql("SELECT DISTINCT l.customer_id, o.stock_code FROM letzte l JOIN orders_clean o ON o.invoice = l.invoice").df()

def trefferquote(recs):
    lang = recs.melt(id_vars="customer_id", value_vars=["rec_1", "rec_2", "rec_3"], value_name="stock_code")
    treffer = lang.merge(versteckt, on=["customer_id", "stock_code"])
    return treffer["customer_id"].nunique() / versteckt["customer_id"].nunique()

kunden = pd.DataFrame({"customer_id": versteckt["customer_id"].unique()})
top3_training = duckdb.sql("""SELECT stock_code FROM training GROUP BY stock_code
                              ORDER BY COUNT(DISTINCT customer_id) DESC, stock_code LIMIT 3""").df()["stock_code"].tolist()
beliebt_recs = kunden.assign(rec_1=top3_training[0], rec_2=top3_training[1], rec_3=top3_training[2])
zusammen_recs = kunden.merge(zusammen_gekauft("training"), on="customer_id", how="left")
vergleich = pd.Series({"Beliebt": trefferquote(beliebt_recs), "Zusammen gekauft": trefferquote(zusammen_recs)})
vergleich.plot.barh(title="Trefferquote: mindestens eine Empfehlung in der letzten Bestellung", figsize=(7, 2))
plt.show()
vergleich.round(3)

**Plausibel?** Ist «zusammen gekauft» klar besser als «beliebt»? Eine Trefferquote über 100 % oder fast 0 % heisst: Da stimmt etwas nicht. Und: Wurden die Paare nur aus `training` gezählt? Sonst hat das Verfahren die versteckten Bestellungen schon gesehen (Leakage).

### 5 Speichern

In [ ]:
speichern(empfehlungen, "empfehlungen")

## Erweiterung (optional)

### E1 Ausgelistete Produkte

**Ziel:** Produkte finden, die seit Monaten niemand mehr kauft. Wer sie empfiehlt, verschwendet einen Platz in der E-Mail.

**Prompt für Gemini** (kopieren und im Gemini-Fenster einfügen):
```text
Welche Produkte wurden in orders_clean früher verkauft, aber seit dem 1. Juli 2025 nie mehr? Zeige stock_code, description und das letzte Verkaufsdatum. Wie oft kommen sie in empfehlungen vor?
```

In [ ]:
# Euer Code: fügt hier ein, was Gemini vorschlägt, und führt die Zelle aus (▶).

In [ ]:
#@title 💡 Lösung: nur wenn ihr nicht weiterkommt (Doppelklick zeigt den Code, ▶ führt ihn aus)
ausgelistet = duckdb.sql("""
SELECT o.stock_code, p.description, MAX(o.invoice_date) AS zuletzt_verkauft
FROM orders_clean o JOIN products p ON p.stock_code = o.stock_code
GROUP BY o.stock_code, p.description
HAVING MAX(o.invoice_date) < DATE '2025-07-01'
ORDER BY zuletzt_verkauft
""").df()
display(ausgelistet)
empfehlungen[["rec_1", "rec_2", "rec_3"]].isin(ausgelistet["stock_code"]).sum()

**Plausibel?** Wenn ausgelistete Produkte empfohlen werden: Filtert sie in zusammen_gekauft heraus.

### E2 Schon gekauft ausschliessen

**Ziel:** Prüfen, ob Empfehlungen besser werden, wenn man nur Produkte empfiehlt, die der Kunde noch nicht hat.

**Prompt für Gemini** (kopieren und im Gemini-Fenster einfügen):
```text
Ändere zusammen_gekauft so, dass Produkte, die der Kunde schon gekauft hat, nicht empfohlen werden. Wie verändert sich die Trefferquote aus Schritt 4?
```

In [ ]:
# Euer Code: fügt hier ein, was Gemini vorschlägt, und führt die Zelle aus (▶).

In [ ]:
#@title 💡 Lösung: nur wenn ihr nicht weiterkommt (Doppelklick zeigt den Code, ▶ führt ihn aus)
def zusammen_gekauft_neu(tabelle):
    return duckdb.sql(f"""
    WITH rechnungen AS (SELECT DISTINCT invoice, stock_code FROM {tabelle}),
    paare AS (SELECT a.stock_code AS produkt, b.stock_code AS empfehlung, COUNT(*) AS gemeinsam
              FROM rechnungen a JOIN rechnungen b ON a.invoice = b.invoice AND a.stock_code <> b.stock_code
              GROUP BY a.stock_code, b.stock_code),
    gekauft AS (SELECT DISTINCT customer_id, stock_code FROM {tabelle}),
    punkte AS (SELECT g.customer_id, p.empfehlung, SUM(p.gemeinsam) AS punkte
               FROM gekauft g JOIN paare p ON p.produkt = g.stock_code
               WHERE NOT EXISTS (SELECT 1 FROM gekauft schon      -- schon gekauft: nicht empfehlen
                                 WHERE schon.customer_id = g.customer_id AND schon.stock_code = p.empfehlung)
               GROUP BY g.customer_id, p.empfehlung),
    rang AS (SELECT *, ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY punkte DESC, empfehlung) AS r FROM punkte)
    SELECT customer_id, MAX(CASE WHEN r = 1 THEN empfehlung END) AS rec_1,
           MAX(CASE WHEN r = 2 THEN empfehlung END) AS rec_2, MAX(CASE WHEN r = 3 THEN empfehlung END) AS rec_3
    FROM rang WHERE r <= 3 GROUP BY customer_id
    """).df()

trefferquote(kunden.merge(zusammen_gekauft_neu("training"), on="customer_id", how="left"))

**Plausibel?** Sinkt die Trefferquote? Viele Kunden sind Wiederverkäufer und bestellen dasselbe nochmals. Was ist für eine E-Mail besser: Bekanntes oder Neues? Die Messmethode entscheidet, was «besser» heisst.